# Error Analysis — Where and Why Models Fail

Deep dive into prediction failures:
1. Worst prediction days (high confidence, wrong direction)
2. Error patterns by sentiment regime
3. Prediction confidence distributions
4. Black Swan case study

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

with open("../evaluation/btc_usd_evaluation.json") as f:
    eval_data = json.load(f)

df = pd.read_parquet("../data/processed/btc_usd_model_ready.parquet")

best_model = "lstm_all_features"
best = eval_data[best_model]
dates = pd.to_datetime(best["dates"])
y_true = np.array(best["y_true"])
y_prob = np.array(best["y_prob"])
y_pred = (y_prob > 0.5).astype(int)
correct = y_true == y_pred

print(f"Best model: {best_model}")
print(f"Test period: {dates.min().date()} → {dates.max().date()} ({len(dates)} days)")
print(f"Correct: {correct.sum()} / {len(correct)} ({correct.mean()*100:.1f}%)")

## 1. Prediction Confidence Distribution

How confident is the model, and does confidence correlate with correctness?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Probability distribution: correct vs incorrect
axes[0].hist(y_prob[correct], bins=30, alpha=0.6, color="#2ecc71", label="Correct", density=True)
axes[0].hist(y_prob[~correct], bins=30, alpha=0.6, color="#e74c3c", label="Incorrect", density=True)
axes[0].axvline(x=0.5, color="black", linestyle="--", alpha=0.5)
axes[0].set_xlabel("Predicted P(Up)")
axes[0].set_ylabel("Density")
axes[0].set_title("Prediction Confidence: Correct vs Incorrect")
axes[0].legend()

# Calibration: binned accuracy vs predicted probability
bins_cal = np.linspace(0, 1, 11)
bin_indices = np.digitize(y_prob, bins_cal) - 1
cal_data = []
for i in range(len(bins_cal) - 1):
    mask = bin_indices == i
    if mask.sum() > 0:
        cal_data.append({
            "bin_center": (bins_cal[i] + bins_cal[i+1]) / 2,
            "actual_positive_rate": y_true[mask].mean(),
            "count": mask.sum(),
        })

cal_df = pd.DataFrame(cal_data)
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfect calibration")
axes[1].scatter(cal_df["bin_center"], cal_df["actual_positive_rate"],
               s=cal_df["count"] * 5, color="#3498db", alpha=0.7, edgecolors="white")
axes[1].set_xlabel("Predicted P(Up)")
axes[1].set_ylabel("Actual P(Up)")
axes[1].set_title("Calibration Plot (size = sample count)")
axes[1].legend()

plt.suptitle("Prediction Confidence Analysis", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 2. Worst Prediction Days

Days where the model was most confidently wrong — these reveal systematic blind spots.

In [ ]:
worst = pd.DataFrame(best["worst_predictions"])
print("Top 10 worst predictions (highest confidence errors):\n")
print(worst.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
worst_plot = worst.head(10)
colors_worst = ["#e74c3c" if a == "Down" else "#2ecc71" for a in worst_plot["actual"]]
bars = ax.barh(range(len(worst_plot)), worst_plot["confidence_error"], color=colors_worst)

ax.set_yticks(range(len(worst_plot)))
ax.set_yticklabels([f"{d} ({a})" for d, a in zip(worst_plot["date"], worst_plot["actual"])])
ax.set_xlabel("Confidence Error Score")
ax.set_title("Worst Predictions — Model's Most Confident Misses", fontsize=13, fontweight="bold")
ax.invert_yaxis()

for bar, prob in zip(bars, worst_plot["predicted_prob_up"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"P(Up)={prob:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 3. Error Patterns by Sentiment Regime

Do models fail more during extreme sentiment? This reveals whether the model handles market stress well.

In [ ]:
min_len = min(len(dates), len(y_true))
test_df = df.loc[dates[:min_len]].copy()
test_df["correct"] = correct[:min_len]
test_df["y_prob"] = y_prob[:min_len]

bins = [0, 20, 40, 60, 80, 100]
labels_regime = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
test_df["regime"] = pd.cut(test_df["fng_value"], bins=bins, labels=labels_regime, include_lowest=True)

regime_acc = test_df.groupby("regime", observed=True).agg(
    accuracy=("correct", "mean"),
    n_days=("correct", "count"),
    avg_confidence=("y_prob", lambda x: (x - 0.5).abs().mean()),
).round(4)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors_regime = ["#c0392b", "#e74c3c", "#95a5a6", "#2ecc71", "#27ae60"]

valid_regimes = regime_acc[regime_acc["n_days"] > 0]
axes[0].bar(valid_regimes.index, valid_regimes["accuracy"] * 100, color=colors_regime[:len(valid_regimes)])
axes[0].axhline(y=50, color="gray", linestyle="--", alpha=0.6)
axes[0].set_ylabel("Accuracy %")
axes[0].set_title("Model Accuracy by Sentiment Regime")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(valid_regimes.index, valid_regimes["n_days"], color=colors_regime[:len(valid_regimes)])
axes[1].set_ylabel("Number of Test Days")
axes[1].set_title("Test Days per Regime")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Error Analysis by Fear & Greed Regime", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("Accuracy by regime:")
print(regime_acc.to_string())

## 4. Prediction Timeline

Visual timeline of predictions vs actual outcomes during the test period.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

plot_dates = dates[:min_len]

# Price during test period
if len(test_df) > 0:
    axes[0].plot(test_df.index, test_df["close"], color="#2c3e50", linewidth=1.2)
    axes[0].set_ylabel("BTC Price (USD)")
    axes[0].set_title("BTC Price During Test Period")

# Predicted probability
axes[1].fill_between(plot_dates, 0.5, y_prob[:min_len],
                     where=y_prob[:min_len] >= 0.5, alpha=0.4, color="#2ecc71", label="Predict Up")
axes[1].fill_between(plot_dates, 0.5, y_prob[:min_len],
                     where=y_prob[:min_len] < 0.5, alpha=0.4, color="#e74c3c", label="Predict Down")
axes[1].axhline(y=0.5, color="black", linewidth=0.8)
axes[1].set_ylabel("P(Up)")
axes[1].set_title("Model Predictions")
axes[1].legend()

# Correct / incorrect markers
correct_colors = ["#2ecc71" if c else "#e74c3c" for c in correct[:min_len]]
axes[2].scatter(plot_dates, y_true[:min_len], c=correct_colors, s=20, alpha=0.7)
axes[2].set_ylabel("Actual Direction")
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["Down", "Up"])
axes[2].set_title("Actual Outcomes (green = correct prediction, red = wrong)")
axes[2].set_xlabel("Date")

plt.suptitle(f"Prediction Timeline — {best_model}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Black Swan Case Study

Identify the single largest prediction failure and analyze why the model missed it. This demonstrates critical thinking about model limitations — a key differentiator for portfolio projects.

In [ ]:
if len(worst) > 0:
    black_swan = worst.iloc[0]
    bs_date = pd.to_datetime(black_swan["date"])

    print("=" * 60)
    print("BLACK SWAN CASE STUDY")
    print("=" * 60)
    print(f"\nDate:               {black_swan['date']}")
    print(f"Actual direction:   {black_swan['actual']}")
    print(f"Model P(Up):        {black_swan['predicted_prob_up']:.4f}")
    print(f"Daily return:       {black_swan['daily_return']}%")
    print(f"Fear & Greed:       {black_swan['fng_value']}")
    print(f"Volatility (20d):   {black_swan['volatility_20d']}%")

    # Context: surrounding days
    if bs_date in df.index:
        window_start = max(0, df.index.get_loc(bs_date) - 5)
        window_end = min(len(df), df.index.get_loc(bs_date) + 6)
        context = df.iloc[window_start:window_end][["close", "daily_return", "fng_value", "volatility_20d"]].copy()
        context["daily_return"] = (context["daily_return"] * 100).round(2)
        context["volatility_20d"] = (context["volatility_20d"] * 100).round(2)
        context.index = context.index.strftime("%Y-%m-%d")
        print(f"\nSurrounding context (±5 days):")
        print(context.to_string())

    print(f"""
Analysis:
- The model predicted P(Up) = {black_swan['predicted_prob_up']:.3f} but the market went {black_swan['actual']}.
- Fear & Greed was at {black_swan['fng_value']} — {"fear territory" if black_swan['fng_value'] < 40 else "neutral" if black_swan['fng_value'] < 60 else "greed territory"}.
- This type of failure is common when external events (regulatory news, exchange hacks,
  macroeconomic shocks) drive sudden reversals that sentiment indices haven't yet captured.
- The Fear & Greed Index updates daily — it cannot react to intraday breaking news.
- Mitigation strategies: real-time news monitoring, shorter sentiment windows, or
  incorporating a volatility-based "uncertainty" flag to reduce confidence during
  high-volatility regimes.
""")

## 6. Error Rate Over Time

Does the model degrade over time? A rising error rate in the most recent period would suggest concept drift.

In [ ]:
rolling_accuracy = pd.Series(correct[:min_len].astype(float), index=plot_dates).rolling(20).mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rolling_accuracy.index, rolling_accuracy * 100, color="#3498db", linewidth=1.5)
ax.axhline(y=50, color="red", linestyle="--", alpha=0.5, label="Random baseline (50%)")
ax.fill_between(rolling_accuracy.index, 50, rolling_accuracy * 100,
                where=rolling_accuracy * 100 >= 50, alpha=0.2, color="#2ecc71")
ax.fill_between(rolling_accuracy.index, 50, rolling_accuracy * 100,
                where=rolling_accuracy * 100 < 50, alpha=0.2, color="#e74c3c")
ax.set_ylabel("Accuracy %")
ax.set_xlabel("Date")
ax.set_title("20-Day Rolling Accuracy (test period)", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()